## Базовые эксперименты и подбор моделей для задачи анализа тональности
Ноутбук проводит полный цикл экспериментов по обучению и оценке нескольких текстовых моделей: TF-IDF + LogisticRegression (с гиперпараметрической оптимизацией через Optuna), TF-IDF + LinearSVC и TF-IDF + SVD + HistGradientBoosting. В файле загружаются подготовленные данные, обучаются модели с разными конфигурациями признаков, сравниваются результаты на валидации, проводится анализ ошибок, а лучшие пайплайны сохраняются для дальнейшего анализа.

#### Результаты классических моделей (TF-IDF + ML) на валидации
В рамках экспериментов были протестированы три классические пайплайна: Logistic Regression с гипероптимизацией Optuna, LinearSVC и модель HistGradientBoosting поверх SVD-сжатых TF-IDF-признаков. Все модели обучались на подготовленных текстах с символьным TF-IDF (char-level) и оценивались на единой валидационной выборке. Итоги показывают умеренное качество с разбросом macro-F1 в диапазоне **0.60–0.64**, при этом лучший результат достигнут Logistic Regression в процессе поиска гиперпараметров (но после дообучения на полном train качество слегка просело).

**Итоговые macro-F1:**
- **TF-IDF + Logistic Regression (Optuna best):** 0.6378 (в ходе поиска),  
  **0.6222** — после финального обучения на полном train  
- **TF-IDF + LinearSVC:** **0.5997** — стабильный, но самый слабый результат  
- **TF-IDF + SVD + HistGradientBoosting:** **0.6101** — немного лучше SVC

**Вывод:** Среди классических моделей наиболее сильным остаётся TF-IDF + LogisticRegression c оптимизированными параметрами, однако даже она заметно уступает BERT-моделям. Эти результаты подтверждают, что классические методы могут служить хорошей «базовой линией», но не являются оптимальными для задачи.



In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import os

# Пути к твоим файлам
TRAIN_PATH = "../data/processed/train_super.csv"
VAL_PATH   = "../data/processed/val_super.csv"
MODEL_DIR  = "./model"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Читаем данные...")
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

print("Форма train:", train_df.shape)
print("Форма val:  ", val_df.shape)

Читаем данные...
Форма train: (106311, 4)
Форма val:   (11771, 4)


In [3]:
X_train_full = train_df["text"].astype(str)
y_train_full = train_df["label"]
X_val = val_df["text"].astype(str)
y_val = val_df["label"]

train_sample = train_df.sample(20000, random_state=42)
X_train = train_sample["text"].astype(str)
y_train = train_sample["label"]

## Оптимизация модели (TF-IDF + Logistic Regression) с помощью Optuna
Скрипт загружает обучающие данные, формирует подвыборку для ускорения поиска гиперпараметров и запускает Optuna для подбора параметров TF-IDF и логистической регрессии. После нахождения оптимальной конфигурации модель переобучается на полном датасете, оценивается на валидации и сохраняется как готовый пайплайн.


In [ ]:
import os
import warnings

import optuna
import pandas as pd
from joblib import dump
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

TRAIN_PATH = "../data/processed/train_super.csv"
VAL_PATH   = "../data/processed/val_super.csv"
MODEL_DIR  = "./model"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Читаем данные...")
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

print("Форма train:", train_df.shape)
print("Форма val:  ", val_df.shape)

X_train_full = train_df["text"].astype(str)
y_train_full = train_df["label"]

X_val = val_df["text"].astype(str)
y_val = val_df["label"]

sample_frac = 0.3
train_sample = train_df.sample(frac=sample_frac, random_state=42)

X_train = train_sample["text"].astype(str)
y_train = train_sample["label"]

print(f"Для Optuna используем {len(X_train)} объектов (из {len(X_train_full)})")


# === Целевая функция для Optuna ===
def objective(trial: optuna.Trial) -> float:
    # ----- TF-IDF -----
    analyzer = trial.suggest_categorical("tfidf_analyzer", ["word", "char"])

    if analyzer == "word":
        ngram_range = trial.suggest_categorical(
            "tfidf_ngram_range_word",
            [(1, 1), (1, 2), (1, 3)],
        )
    else:  # char
        ngram_range = trial.suggest_categorical(
            "tfidf_ngram_range_char",
            [(3, 5), (3, 6), (3, 7)],
        )

    max_features = trial.suggest_categorical(
        "tfidf_max_features",
        [30_000, 50_000, 80_000, 120_000],
    )
    min_df = trial.suggest_categorical(
        "tfidf_min_df",
        [2, 5, 10],
    )
    max_df = trial.suggest_categorical(
        "tfidf_max_df",
        [0.9, 0.95, 1.0],
    )
    sublinear_tf = trial.suggest_categorical(
        "tfidf_sublinear_tf",
        [True, False],
    )
    norm = trial.suggest_categorical(
        "tfidf_norm",
        ["l2", None],
    )
    use_idf = trial.suggest_categorical(
        "tfidf_use_idf",
        [True, False],
    )

    tfidf = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=sublinear_tf,
        norm=norm,
        use_idf=use_idf,
    )

    # ----- LogisticRegression -----
    solver = trial.suggest_categorical("lr_solver", ["lbfgs", "saga"])

    if solver == "lbfgs":
        penalty = "l2"
        l1_ratio = None
    else:  # saga
        penalty = trial.suggest_categorical(
            "lr_penalty",
            ["l1", "l2", "elasticnet"],
        )
        if penalty == "elasticnet":
            l1_ratio = trial.suggest_float("lr_l1_ratio", 0.0, 1.0)
        else:
            l1_ratio = None

    C = trial.suggest_loguniform("lr_C", 1e-3, 1e2)
    class_weight = trial.suggest_categorical(
        "lr_class_weight",
        [None, "balanced"],
    )
    fit_intercept = trial.suggest_categorical(
        "lr_fit_intercept",
        [True, False],
    )

    clf = LogisticRegression(
        solver=solver,
        penalty=penalty,
        C=C,
        class_weight=class_weight,
        fit_intercept=fit_intercept,
        l1_ratio=l1_ratio,
        max_iter=1000,
        n_jobs=-1,
    )

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("clf", clf),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    f1_macro = f1_score(y_val, y_pred, average="macro")

    return f1_macro



# === Запуск Optuna ===
study_name = "logreg_tfidf_study"
study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
)

N_TRIALS = 5  
print(f"Запускаем Optuna на {N_TRIALS} трейлов...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nЛучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"Лучший macro-F1 (val) во время поиска: {study.best_value:.4f}")

best_params = study.best_params

# TF-IDF
analyzer = best_params["tfidf_analyzer"]

if analyzer == "word":
    ngram_range = best_params["tfidf_ngram_range_word"]
else:
    ngram_range = best_params["tfidf_ngram_range_char"]

max_features = best_params["tfidf_max_features"]
min_df = best_params["tfidf_min_df"]
max_df = best_params["tfidf_max_df"]
sublinear_tf = best_params["tfidf_sublinear_tf"]
norm = best_params["tfidf_norm"]

tfidf_best = TfidfVectorizer(
    analyzer=analyzer,
    ngram_range=ngram_range,
    max_features=max_features,
    min_df=min_df,
    max_df=max_df,
    sublinear_tf=sublinear_tf,
    norm=norm,
)


# LogisticRegression
solver = best_params["lr_solver"]
if solver == "lbfgs":
    penalty = "l2"
    l1_ratio = None
else:
    penalty = best_params["lr_penalty"]
    l1_ratio = best_params.get("lr_l1_ratio", None)

C = best_params["lr_C"]
class_weight = best_params["lr_class_weight"]
fit_intercept = best_params["lr_fit_intercept"]

clf_best = LogisticRegression(
    solver=solver,
    penalty=penalty,
    C=C,
    class_weight=class_weight,
    fit_intercept=fit_intercept,
    l1_ratio=l1_ratio,
    max_iter=1000,
    n_jobs=-1,
)

best_pipeline = Pipeline([
    ("tfidf", tfidf_best),
    ("clf", clf_best),
])

print("\nОбучаем лучшую конфигурацию на ПОЛНОМ train...")
best_pipeline.fit(X_train_full, y_train_full)

# Оценка на val
y_pred_final = best_pipeline.predict(X_val)
final_f1_macro = f1_score(y_val, y_pred_final, average="macro")

print(f"\nИтоговый macro-F1 на val после дообучения на всём train: {final_f1_macro:.4f}")
print("\n=== Classification report (final best pipeline) ===")
print(classification_report(y_val, y_pred_final, digits=4))


pipeline_path = os.path.join(MODEL_DIR, "sentiment_lr_optuna.joblib")
dump(best_pipeline, pipeline_path)
print(f"\n✅ Оптимизированный пайплайн (TF-IDF + LogisticRegression + Optuna) сохранён в {pipeline_path}")


/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Читаем данные...


[I 2025-11-29 13:15:54,574] A new study created in memory with name: logreg_tfidf_study


Форма train: (106311, 4)
Форма val:   (11771, 4)
Для Optuna используем 31893 объектов (из 106311)
Запускаем Optuna на 5 трейлов...


  0%|          | 0/5 [00:00<?, ?it/s]/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 5) which is of type tuple.
  warnings.warn(message)
/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  warnings.warn(message)
/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 7) which is of type tuple.
  warnings.warn(message)
Best trial: 0. Best value: 0.63496:  20%|██        | 1/5 [00:19<01:16, 19.05s/

...

[I 2025-11-29 13:17:02,439] Trial 2 finished with value: 0.6337161068061303 and parameters: {'tfidf_analyzer': 'word', 'tfidf_ngram_range_word': (1, 1), 'tfidf_max_features': 50000, 'tfidf_min_df': 5, 'tfidf_max_df': 0.9, 'tfidf_sublinear_tf': False, 'tfidf_norm': None, 'tfidf_use_idf': False, 'lr_solver': 'lbfgs', 'lr_C': 0.1533195429919741, 'lr_class_weight': None, 'lr_fit_intercept': True}. Best is trial 1 with value: 0.6377601134206201.


Best trial: 1. Best value: 0.63776:  80%|████████  | 4/5 [01:19<00:16, 16.97s/it]/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  warnings.warn(message)
/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  warnings.warn(message)
/Users/macbook/MosHack-Change_2025/.venv/lib/python3.13/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  warnings.warn(message)


[I 2025-11-29 13:17:14,167] Trial 3 finished with value: 0.6300940641653622 and parameters: {'tfidf_analyzer': 'word', 'tfidf_ngram_range_word': (1, 3), 'tfidf_max_features': 120000, 'tfidf_min_df': 2, 'tfidf_max_df': 0.95, 'tfidf_sublinear_tf': False, 'tfidf_norm': None, 'tfidf_use_idf': False, 'lr_solver': 'lbfgs', 'lr_C': 38.66257213529479, 'lr_class_weight': None, 'lr_fit_intercept': True}. Best is trial 1 with value: 0.6377601134206201.


Best trial: 1. Best value: 0.63776: 100%|██████████| 5/5 [01:23<00:00, 16.75s/it]


[I 2025-11-29 13:17:18,324] Trial 4 finished with value: 0.6228674466043086 and parameters: {'tfidf_analyzer': 'word', 'tfidf_ngram_range_word': (1, 1), 'tfidf_max_features': 80000, 'tfidf_min_df': 10, 'tfidf_max_df': 0.95, 'tfidf_sublinear_tf': True, 'tfidf_norm': 'l2', 'tfidf_use_idf': False, 'lr_solver': 'lbfgs', 'lr_C': 20.243041936257313, 'lr_class_weight': 'balanced', 'lr_fit_intercept': True}. Best is trial 1 with value: 0.6377601134206201.

Лучшие параметры:
  tfidf_analyzer: char
  tfidf_ngram_range_char: (3, 5)
  tfidf_max_features: 30000
  tfidf_min_df: 2
  tfidf_max_df: 0.95
  tfidf_sublinear_tf: False
  tfidf_norm: None
  tfidf_use_idf: False
  lr_solver: lbfgs
  lr_C: 0.01899479531532875
  lr_class_weight: None
  lr_fit_intercept: False
Лучший macro-F1 (val) во время поиска: 0.6378

Обучаем лучшую конфигурацию на ПОЛНОМ train...

Итоговый macro-F1 на val после дообучения на всём train: 0.6222

=== Classification report (final best pipeline) ===
              precision    

ИСПОЛЬЗОВАНИЕ ЗАГРУЖЕННОЙ МОДЕЛИ TF-IDF + LogisticRegression

In [2]:
from joblib import load

pipeline = load("./model/sentiment_lr_optuna.joblib")

def preprocess_text(text: str, src: str | None = None) -> str:
    if src is not None:
        return f"[SRC_{src}] {text}"
    return text

def predict_sentiment(text: str, src: str | None = None):
    text_proc = preprocess_text(text, src)
    pred = pipeline.predict([text_proc])[0]
    return int(pred)
print(predict_sentiment("Все очень очень очень очень плохо!"))

2


In [3]:
import pandas as pd
from sklearn.metrics import confusion_matrix

y_pred = best_pipeline.predict(X_val)
cm = confusion_matrix(y_val, y_pred)
print(cm)

errors = val_df.copy()
errors["y_true"] = y_val
errors["y_pred"] = y_pred
errs = errors[errors["y_true"] != errors["y_pred"]]

# посмотреть по 10 примеров каждой пары (true -> pred)
for t, p in [(0,1), (0,2), (1,0), (1,2), (2,0), (2,1)]:
    print(f"\n=== true={t}, pred={p} ===")
    print(errs[(errs["y_true"] == t) & (errs["y_pred"] == p)]["text"].head(10).tolist())

[[2077  806 1013]
 [ 825 2814  298]
 [1151  357 2430]]

=== true=0, pred=1 ===
['[SRC_kinopoisk] Отличный фильм. Мне очень понравился. Вроде и спецэффектов зрелищных нет, и сюжет не особо насыщен событиями, но фильм смотрел не отрываясь.В картине нет ничего лишнего, актёры превосходно играют свои роли. А роли надо сказать достаточно нетривиальные. Помимо в сего прочего в фильме много философии, но не слишком грузящей. Несмотря на криминальный сюжет, фильм на мой взгляд получился добрый.Вобщем, советую посмотреть всем.', "[SRC_kinopoisk] «Безумный Макс: Дорога ярости» – фильм, которую поклонники франшизы ждали 30 лет и надо сказать, оно того стоило. За эти долгие годы, «Безумный Макс» потерпел немало изменений, в частности ушел Мел Гибсон, который играл роль Макса Рокатански, но его место занял не менее великолепный Том Харди.Вскоре после отмщения за смерть жены и сына, Макс Рокатански покинул ряды «Основного силового патруля» и уехал в глушь, где скитается в одиночестве, пока мир медле

### Обучение модели TF-IDF + LinearSVC на подвыборке данных
Функция `train_svc_best` строит облегчённый пайплайн TF-IDF по символам и LinearSVC, позволяющий быстро обучать модель даже на части датасета. После обучения на подвыборке train-данных модель проверяется на валидации, вычисляется macro-F1, и готовый пайплайн сохраняется на диск.


In [ ]:
import os
from typing import Tuple, Dict, Optional

import pandas as pd
from joblib import dump
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score


def train_svc_best(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    model_dir: str = "./model",
    model_name: str = "sentiment_svc_best.joblib",
    sample_frac: Optional[float] = 0.4, 
) -> Tuple[Pipeline, Dict[str, float]]:
    """
    Обучает LinearSVC + TF-IDF(char) и сохраняет пайплайн.
    Можно обучать не на всём train, а на подвыборке (sample_frac), чтобы ускорить обучение.
    """

    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, model_name)

    # --- Подвыборка train для обучения SVC ---
    if sample_frac is not None and sample_frac < 1.0:
        train_sample = train_df.sample(frac=sample_frac, random_state=42)
        print(f"Обучаем SVC на подвыборке: {len(train_sample)} из {len(train_df)}")
    else:
        train_sample = train_df
        print(f"Обучаем SVC на всём train: {len(train_sample)} объектов")

    X_train = train_sample["text"].astype(str)
    y_train = train_sample["label"]

    X_val = val_df["text"].astype(str)
    y_val = val_df["label"]

    # --- TF-IDF  ---
    tfidf = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 4),   # было (3,5) — чуть упрощаем
        max_features=20_000,  # было 30_000 — уменьшаем размер
        min_df=2,
        max_df=0.95,
        sublinear_tf=False,
        norm=None,
        use_idf=False,
    )

    clf = LinearSVC(
        C=0.1,
        loss="squared_hinge",
        class_weight=None,
        random_state=42,
        max_iter=3000,  
    )

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("clf", clf),
    ])

    print("Обучаем TF-IDF(char) + LinearSVC (ускоренный вариант)...")
    pipeline.fit(X_train, y_train)

    # --- Оценка на валидации ---
    y_pred = pipeline.predict(X_val)
    macro_f1 = f1_score(y_val, y_pred, average="macro")

    print("\n=== Результаты SVC на валидации ===")
    print(classification_report(y_val, y_pred, digits=4))
    print(f"macro-F1: {macro_f1:.4f}")

    dump(pipeline, model_path)
    print(f"\nПайплайн (TF-IDF + LinearSVC) сохранён в {model_path}")

    metrics = {"macro_f1": macro_f1}
    return pipeline, metrics


In [ ]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train_super.csv")
val_df   = pd.read_csv("../data/processed/val_super.csv")

pipeline_svc, metrics_svc = train_svc_best(train_df, val_df)
print("macro-F1 SVC:", metrics_svc["macro_f1"])

Обучаем SVC на подвыборке: 42524 из 106311
Обучаем TF-IDF(char) + LinearSVC (ускоренный вариант)...

=== Результаты SVC на валидации ===
              precision    recall  f1-score   support

           0     0.5000    0.5000    0.5000      3896
           1     0.6775    0.6995    0.6883      3937
           2     0.6210    0.6008    0.6107      3938

    accuracy                         0.6005     11771
   macro avg     0.5995    0.6001    0.5997     11771
weighted avg     0.5998    0.6005    0.6000     11771

macro-F1: 0.5997

Пайплайн (TF-IDF + LinearSVC) сохранён в ./model/sentiment_svc_best.joblib
macro-F1 SVC: 0.5996887243513506


In [ ]:
# Инференс

def preprocess_text(text: str, src: str | None = None) -> str:
    if src is not None:
        return f"[SRC_{src}] {text}"
    return text

def predict_sentiment(text: str, src: str | None = None):
    text_proc = preprocess_text(text, src)
    pred = pipeline_svc.predict([text_proc])[0]
    return int(pred)

print(predict_sentiment("Хорошо"))

1


### Обучение модели TF-IDF + SVD + HistGradientBoosting с фиксированными гиперпараметрами
Функция `train_svd_hgb_best` строит пайплайн из символьного TF-IDF, снижения размерности через TruncatedSVD и классификатора HistGradientBoostingClassifier с заранее подобранными параметрами. Модель обучается на train-данных, оценивается на валидационной выборке по macro-F1 и другим метрикам, после чего пайплайн и результаты сохраняются для дальнейшего испо_


In [ ]:
import os
from typing import Tuple, Dict

import pandas as pd
from joblib import dump
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score


def train_svd_hgb_best(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    model_dir: str = "./model",
    model_name: str = "sentiment_svd_hgb_best.joblib",
) -> Tuple[Pipeline, Dict[str, float]]:
    """
    Обучает пайплайн:
        TF-IDF (char-level) -> TruncatedSVD -> HistGradientBoostingClassifier
    с заранее заданными гиперпараметрами и сохраняет модель на диск.

    Ожидает, что в train_df и val_df есть колонки "text" и "label".
    """

    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, model_name)

    X_train = train_df["text"].astype(str)
    y_train = train_df["label"]

    X_val = val_df["text"].astype(str)
    y_val = val_df["label"]

    # --- TF-IDF (char 3–5) с теми же идеями, что и для лучшей LogReg ---
    tfidf = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        max_features=30_000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=False,
        norm=None,
        use_idf=False,
    )

    # --- Снижение размерности TF-IDF через SVD (LSA) ---
    svd = TruncatedSVD(
        n_components=200,   
        random_state=42,
    )

    clf = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_depth=6,       
        max_iter=200,      
        random_state=42,
    )

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("svd", svd),
        ("clf", clf),
    ])

    print("Обучаем TF-IDF(char) + SVD + HistGradientBoosting с фиксированными гиперпараметрами...")
    pipeline.fit(X_train, y_train)

    # --- Оценка на валидации ---
    y_pred = pipeline.predict(X_val)
    macro_f1 = f1_score(y_val, y_pred, average="macro")

    print("\n=== Результаты SVD+HGB на валидации ===")
    print(classification_report(y_val, y_pred, digits=4))
    print(f"macro-F1: {macro_f1:.4f}")

    dump(pipeline, model_path)
    print(f"\nПайплайн (TF-IDF + SVD + HistGradientBoosting) сохранён в {model_path}")

    metrics = {
        "macro_f1": macro_f1,
    }
    return pipeline, metrics


In [16]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train_super.csv")
val_df   = pd.read_csv("../data/processed/val_super.csv")

pipeline_svd_hgb, metrics_svd_hgb = train_svd_hgb_best(train_df, val_df)
print("macro-F1 SVD+HGB:", metrics_svd_hgb["macro_f1"])

def predict_sentiment_svd_hgb(text: str) -> int:
    return int(pipeline_svd_hgb.predict([text])[0])

print(predict_sentiment_svd_hgb("Все очень очень очень очень плохо!"))

Обучаем TF-IDF(char) + SVD + HistGradientBoosting с фиксированными гиперпараметрами...

=== Результаты SVD+HGB на валидации ===
              precision    recall  f1-score   support

           0     0.5113    0.5231    0.5171      3896
           1     0.7016    0.6810    0.6912      3937
           2     0.6201    0.6242    0.6221      3938

    accuracy                         0.6097     11771
   macro avg     0.6110    0.6094    0.6101     11771
weighted avg     0.6114    0.6097    0.6105     11771

macro-F1: 0.6101

Пайплайн (TF-IDF + SVD + HistGradientBoosting) сохранён в ./model/sentiment_svd_hgb_best.joblib
macro-F1 SVD+HGB: 0.6101353764779457
0
